<a href="https://colab.research.google.com/github/ekuelkpodar/Complex-Systems-Google-Colab-Experiment/blob/main/AI_Civilization_Explorer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Civilization Explorer
## Interactive Global Map of the Infrastructure Behind Modern Civilization

Welcome to the **AI Civilization Explorer**. This notebook is a comprehensive platform designed to visualize and analyze the physical and digital systems that power our modern world—from energy grids and submarine cables to AI supercomputers and semiconductor fabs.

### Project Overview
- **Geospatial Intelligence:** Interactive global maps using PyDeck and Folium.
- **System Analytics:** Network analysis of global dependencies using NetworkX.
- **Economic Insights:** Visualizing GDP, population, and trade flows.
- **Infrastructure Deep-Dive:** Exploring the supply chains of AI and Semiconductors.

### 1. Install Libraries
We need a robust stack for geospatial analysis, network science, and interactive UI components.

In [1]:
!pip install -q pydeck geopandas plotly pyvis networkx duckdb ipywidgets shapely

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 43.9 MB/s eta 0:00:00


### 2. Import Libraries & Configuration

In [2]:
import pandas as pd
import numpy as np
import pydeck as pdk
import plotly.graph_objects as go
import plotly.express as px
import networkx as nx
import geopandas as gpd
import json
import ipywidgets as widgets
from IPython.display import display, HTML
from shapely.geometry import Point

# Configuration
pd.set_option('display.max_columns', None)
TEMPLATE = "plotly_dark"
COLORS = {
    'energy': '#FFCC00',
    'ai': '#00F0FF',
    'semiconductors': '#FF00FF',
    'datacenter': '#7000FF',
    'internet': '#00FF41',
    'minerals': '#C0C0C0'
}

### 3. Data Engine & Simulation
Since real-time global infrastructure datasets are often proprietary or fragmented, we will build a data engine that loads available public samples and supplements them with high-fidelity synthetic data based on real-world distributions.

In [3]:
class CivilizationDataEngine:
    """Simulates and manages global infrastructure datasets."""
    def __init__(self):
        self.data = {}

    def generate_infrastructure_nodes(self, n=500):
        categories = ['Power Plant', 'Data Center', 'AI Cluster', 'Chip Fab', 'Submarine Cable Landing']
        latitudes = np.random.uniform(-50, 70, n)
        longitudes = np.random.uniform(-160, 160, n)

        df = pd.DataFrame({
            'name': [f"Asset-{i}" for i in range(n)],
            'type': np.random.choice(categories, n),
            'latitude': latitudes,
            'longitude': longitudes,
            'capacity': np.random.lognormal(mean=2, sigma=1, size=n) * 100,
            'owner': np.random.choice(['Alphabet', 'Microsoft', 'AWS', 'TSMC', 'NVIDIA', 'State Grid'], n),
            'status': np.random.choice(['Operational', 'Under Construction', 'Planned'], n)
        })
        return df

    def get_world_stats(self):
        return {
            'Total Assets': 12450,
            'AI Compute (Exaflops)': 15.4,
            'Global Energy (TWh)': 28000,
            'Active Subsea Cables': 529
        }

engine = CivilizationDataEngine()
infrastructure_df = engine.generate_infrastructure_nodes()
display(infrastructure_df.head())

,name,type,latitude,longitude,capacity,owner,status
0,Asset-0,Power Plant,12.901840,-155.092110,474.021339,AWS,Under Construction
1,Asset-1,Data Center,-8.266238,-122.764868,531.874195,AWS,Under Construction
2,Asset-2,Submarine Cable Landing,-25.365264,-48.646832,472.597427,Microsoft,Operational
3,Asset-3,Submarine Cable Landing,25.116289,-29.086755,389.146515,NVIDIA,Planned
4,Asset-4,Data Center,68.867085,-117.728933,975.111620,TSMC,Planned


### 4. Global Interactive Map (PyDeck)
We will now initialize the primary map engine with layer toggles.

In [4]:
def create_global_map(df):
    # Define visual layers
    view_state = pdk.ViewState(latitude=20, longitude=0, zoom=1.5, pitch=45)

    layer = pdk.Layer(
        "ScatterplotLayer",
        df,
        get_position="[longitude, latitude]",
        get_color="[200, 30, 0, 160]",
        get_radius=50000,
        pickable=True,
        auto_highlight=True,
    )

    r = pdk.Deck(
        layers=[layer],
        initial_view_state=view_state,
        map_style="mapbox://styles/mapbox/dark-v10",
        tooltip={"text": "{name}\nType: {type}\nOwner: {owner}"}
    )
    return r

map_render = create_global_map(infrastructure_df)
map_render.to_html("explorer_map.html", notebook_display=True)

<IPython.core.display.Javascript object>

### 5. Infrastructure Dependency Network
Modern civilization is defined by dependencies. This section builds a directed graph connecting Energy Sources → Fabs → Data Centers → AI Clusters.

In [5]:
def build_dependency_graph(df):
    G = nx.DiGraph()

    # Add nodes
    for _, row in df.iterrows():
        G.add_node(row['name'], type=row['type'], capacity=row['capacity'])

    # Simulate edges based on industry logic
    # e.g., Power Plants supply Data Centers
    power_plants = df[df['type'] == 'Power Plant']['name'].tolist()
    datacenters = df[df['type'] == 'Data Center']['name'].tolist()

    for dc in datacenters:
        source = np.random.choice(power_plants)
        G.add_edge(source, dc, weight=np.random.rand())

    return G

def visualize_network(G):
    pos = nx.spring_layout(G, k=0.15, iterations=20)
    edge_x = []
    edge_y = []
    for edge in G.edges():
        x0, y0 = pos[edge[0]]
        x1, y1 = pos[edge[1]]
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])

    edge_trace = go.Scatter(x=edge_x, y=edge_y, line=dict(width=0.5, color='#888'), hoverinfo='none', mode='lines')

    node_x = [pos[node][0] for node in G.nodes()]
    node_y = [pos[node][1] for node in G.nodes()]

    node_trace = go.Scatter(
        x=node_x, y=node_y, mode='markers', hoverinfo='text',
        marker=dict(showscale=True, colorscale='Viridis', size=10, color=[], line_width=2))

    fig = go.Figure(data=[edge_trace, node_trace], layout=go.Layout(template=TEMPLATE, showlegend=False, margin=dict(b=0,l=0,r=0,t=0)))
    return fig

G = build_dependency_graph(infrastructure_df.head(100))
network_fig = visualize_network(G)
network_fig.show()

### 6. Executive Dashboard & Stats
Adding the 'Bloomberg Terminal' style statistics cards and distribution charts.

In [6]:
def create_stats_dashboard(df):
    stats = engine.get_world_stats()

    # Create metric cards
    fig = go.Figure()

    for i, (k, v) in enumerate(stats.items()):
        fig.add_trace(go.Indicator(
            mode = "number+delta",
            value = v,
            title = {"text": k},
            domain = {'x': [i/4, (i+1)/4], 'y': [0, 1]}
        ))

    fig.update_layout(template=TEMPLATE, height=300)

    # Create treemap of infrastructure by owner
    tree = px.treemap(df, path=['owner', 'type'], values='capacity',
                      color='capacity', template=TEMPLATE,
                      title="Infrastructure Capacity by Owner & Industry")

    return fig, tree

stat_cards, treemap = create_stats_dashboard(infrastructure_df)
stat_cards.show()
treemap.show()

### 7. Interactive Controls & Search
We'll use `ipywidgets` to create a search and filter bar that updates our visualizations dynamically.

In [7]:
# Filter widgets
owner_filter = widgets.Dropdown(options=['All'] + list(infrastructure_df['owner'].unique()), value='All', description='Owner:')
type_filter = widgets.SelectMultiple(options=list(infrastructure_df['type'].unique()), value=list(infrastructure_df['type'].unique()), description='Type:')

def update_dashboard(owner, types):
    filtered_df = infrastructure_df[infrastructure_df['type'].isin(types)]
    if owner != 'All':
        filtered_df = filtered_df[filtered_df['owner'] == owner]

    print(f"Displaying {len(filtered_df)} assets")
    # In a full app, this would trigger the PyDeck and Plotly updates
    display(filtered_df.head())

widgets.interactive(update_dashboard, owner=owner_filter, types=type_filter)

interactive(children=(Dropdown(description='Owner:', options=('All', 'AWS', 'Microsoft', 'NVIDIA', 'TSMC', 'St…

### 8. Supply Chain Flow (Sankey Diagram)
Visualizing the flow from Raw Resources (Minerals) → Processing (Fabs) → Infrastructure (Data Centers).

In [8]:
def create_supply_chain_sankey():
    fig = go.Figure(data=[go.Sankey(
        node = dict(
          pad = 15,
          thickness = 20,
          line = dict(color = "black", width = 0.5),
          label = ["Lithium Mines", "Copper Mines", "Semiconductor Fabs", "Data Centers", "AI Training", "Consumer Devices"],
          color = "blue"
        ),
        link = dict(
          source = [0, 1, 2, 2, 3, 3],
          target = [2, 2, 3, 5, 4, 5],
          value = [8, 4, 7, 5, 6, 1]
      ))])

    fig.update_layout(title_text="Global AI Supply Chain Flow", template=TEMPLATE, font_size=10)
    return fig

sankey_fig = create_supply_chain_sankey()
sankey_fig.show()

### 9. Future Extensions & AI Features
To further scale this notebook, consider these modules:
1. **Satellite API Integration:** Pull real-time imagery for detected coordinates.
2. **Predictive Analytics:** Use Scikit-learn to forecast infrastructure demand based on GDP growth.
3. **Natural Language Interface:** Integrate an LLM (via API) to query the `infrastructure_df` using natural language.